# PDF Form Filling

This notebook shows how to write trusted Pydantic data back into PDFs with DocuFlow.

It covers:

- Filling real AcroForm fields.
- Filling two generated static PDFs: one with obvious blank areas and one with harder layout gaps.
- Filling static PDFs with explicit coordinates.
- Opting into heuristic blank-space detection when the blanks are clear.
- Opting into LLM-assisted blank-space detection when the layout is ambiguous.
- Reading the dedicated `FillingResult` object.


## Setup

Install the form filling dependencies first:

```bash
uv pip install -e ".[forms,llm,jupyter]"
```

The LLM section is skipped unless you explicitly set `RUN_LLM_FORM_DETECTION=1` and configure the provider API key for your selected model.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from pydantic import BaseModel, Field

from docuflow import fill_pdf_form_async
from docuflow.filling import FillingResult

try:
    from pypdf import PdfReader
    from reportlab.pdfgen import canvas
except ImportError as exc:
    raise RuntimeError(
        "Install form dependencies first: uv pip install -e '.[forms,jupyter]'"
    ) from exc


cwd = Path.cwd()
EXAMPLES_DIR = cwd if cwd.name == "examples" else cwd / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "data" / "form_filling_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

In [ ]:
PAGE_SIZE = (612, 792)  # US letter, PDF points


def create_acroform_pdf(path: Path) -> None:
    c = canvas.Canvas(str(path), pagesize=PAGE_SIZE)
    c.setFont("Helvetica", 11)
    c.drawString(72, 720, "Claimant Name")
    c.acroForm.textfield(name="claimant.name", x=180, y=710, width=220, height=22)
    c.drawString(72, 680, "Policy Number")
    c.acroForm.textfield(name="policy_number", x=180, y=670, width=180, height=22)
    c.drawString(72, 640, "I accept the terms")
    c.acroForm.checkbox(name="accepted_terms", x=180, y=636, buttonStyle="check")
    c.save()


def create_easy_static_pdf(path: Path) -> None:
    c = canvas.Canvas(str(path), pagesize=PAGE_SIZE)
    c.setFont("Helvetica", 11)
    c.drawString(72, 720, "Claimant Name:")
    c.line(170, 718, 390, 718)
    c.drawString(72, 690, "Policy Number:")
    c.line(170, 688, 340, 688)
    c.drawString(72, 640, "Notes:")
    c.rect(120, 610, 300, 45)
    c.save()


def create_hard_static_pdf(path: Path) -> None:
    c = canvas.Canvas(str(path), pagesize=PAGE_SIZE)
    c.setFont("Helvetica", 10)
    c.drawString(72, 726, "Please complete the claim details below.")
    c.drawString(72, 690, "Claimant name")
    c.drawString(260, 690, "Policy number")
    c.drawString(72, 640, "Additional notes")
    c.drawString(72, 608, "If the space is empty, please write clearly inside the open area.")
    c.rect(196, 680, 210, 18)
    c.rect(350, 680, 170, 18)
    c.rect(72, 560, 448, 34)
    c.save()


def summarize_result(result: FillingResult) -> None:
    print("success:", result.success)
    print("strategy:", result.strategy)
    print("output_path:", result.output_path)
    print("warnings:", result.warnings)
    print("errors:", result.errors)
    print("fields:")
    for name, field in result.fields.items():
        print(
            f"  {name}: value={field.value!r}, target={field.target_name!r}, "
            f"method={field.method!r}, page={field.page_number}, bbox={field.bbox}"
        )
        if field.placement and (field.placement.confidence is not None or field.placement.reason):
            print(
                f"    placement: source={field.placement.source!r}, "
                f"confidence={field.placement.confidence}, reason={field.placement.reason!r}"
            )


## 1. Fill a Real PDF Form

Real PDF forms contain AcroForm fields. This is the most deterministic filling path because the PDF already knows where each field lives.

In [ ]:
class ClaimForm(BaseModel):
    claimant_name: str = Field(alias="claimant.name")
    policy_number: str
    accepted_terms: bool = False


claim_data = ClaimForm(
    **{
        "claimant.name": "Mario Rossi",
        "policy_number": "POL-123456",
        "accepted_terms": True,
    }
)

acroform_pdf = OUTPUT_DIR / "blank_claim_acroform.pdf"
acroform_filled_pdf = OUTPUT_DIR / "filled_claim_acroform.pdf"
create_acroform_pdf(acroform_pdf)

acro_result = await fill_pdf_form_async(
    str(acroform_pdf),
    data=claim_data,
    output_path=str(acroform_filled_pdf),
    strategy="acroform",
)

summarize_result(acro_result)

In [ ]:
# Verify the AcroForm values written into the output PDF.
reader = PdfReader(str(acroform_filled_pdf))
filled_fields = reader.get_fields()
{name: field.get("/V") for name, field in filled_fields.items()}

## 2. Fill a Static PDF with Explicit Coordinates

Static PDFs have visible blanks but no real PDF fields. The most reliable static path is to provide explicit placements.

DocuFlow uses the same coordinate convention used by extraction evidence and highlights: page-local coordinates, top-left origin, usually PDF points.

In [ ]:
easy_static_pdf = OUTPUT_DIR / "blank_claim_static_easy.pdf"
static_explicit_pdf = OUTPUT_DIR / "filled_claim_static_explicit.pdf"
create_easy_static_pdf(easy_static_pdf)

explicit_result = await fill_pdf_form_async(
    str(easy_static_pdf),
    data={
        "claimant_name": "Mario Rossi",
        "policy_number": "POL-123456",
        "notes": "Submitted from the DocuFlow form filling notebook.",
    },
    output_path=str(static_explicit_pdf),
    strategy="overlay",
    field_map={
        "claimant_name": {
            "page_number": 0,
            "bbox": {"x0": 172, "y0": 58, "x1": 390, "y1": 76},
        },
        "policy_number": {
            "page_number": 0,
            "bbox": {"x0": 172, "y0": 88, "x1": 340, "y1": 106},
        },
        "notes": {
            "page_number": 0,
            "bbox": {"x0": 124, "y0": 138, "x1": 418, "y1": 178},
            "font_size": 9,
        },
    },
)

summarize_result(explicit_result)

## 3. Opt Into Heuristic Blank Detection

Blank detection is not active by default. When you pass `detect_blank_spaces=True`, the default `blank_detection_mode="heuristic"` looks for labeled lines, boxes, and underscore blanks. It works well on the easy static PDF below, where the blanks are obvious, but you should still review the generated PDF.

In [ ]:
static_detected_pdf = OUTPUT_DIR / "filled_claim_static_detected.pdf"

detected_result = await fill_pdf_form_async(
    str(easy_static_pdf),
    data={
        "claimant_name": "Mario Rossi",
        "policy_number": "POL-123456",
    },
    output_path=str(static_detected_pdf),
    strategy="overlay",
    detect_blank_spaces=True,
)

summarize_result(detected_result)

## 4. Optional LLM-Assisted Blank Detection

The second generated PDF uses a harder layout with less obvious whitespace. That is where the vision LLM planner becomes useful.

The LLM is only used as a placement planner. It receives page images plus field names, aliases, and descriptions. It does not receive the values to write. The LLM returns relative 0-1 boxes with top-left origin, and DocuFlow converts those into standard `BoundingBox` page coordinates before writing.

This cell is skipped unless `RUN_LLM_FORM_DETECTION=1` is set.

In [ ]:
RUN_LLM = os.getenv("RUN_LLM_FORM_DETECTION") == "1"
MODEL = os.getenv("DOCUFLOW_FORM_MODEL", "openai/gpt-4o")

if not RUN_LLM:
    print(
        "Skipped. Set RUN_LLM_FORM_DETECTION=1 and configure the provider API key "
        "to run LLM-assisted detection."
    )
else:
    hard_static_pdf = OUTPUT_DIR / "blank_claim_static_hard.pdf"
    static_llm_pdf = OUTPUT_DIR / "filled_claim_static_llm.pdf"
    create_hard_static_pdf(hard_static_pdf)
    llm_result = await fill_pdf_form_async(
        str(hard_static_pdf),
        data={
            "claimant_name": "Mario Rossi",
            "policy_number": "POL-123456",
            "notes": "Detected by the vision placement planner.",
        },
        output_path=str(static_llm_pdf),
        strategy="overlay",
        detect_blank_spaces=True,
        blank_detection_mode="llm",  # or "hybrid"
        model=MODEL,
        min_detection_confidence=0.5,
    )
    summarize_result(llm_result)

## 5. Review and Approve Before Saving (Opt-In)

By default `fill_pdf_form_async` writes the output PDF immediately. Because filling writes data *into* a file, a review step has to defer that write until a human approves it.

Pass `review=True` to *prepare* the fill — DocuFlow builds the plan, populates `result.fields`, runs the review heuristics, and reports `needs_review` / `review_reasons` — **without writing `output_path`**. You can then inspect a rendered preview, correct any value or placement with `edit_field()`, and finally `approve()` + `commit_fill_async()` to write the file (or `reject()` to discard it).

This mirrors the extraction review workflow, but is entirely opt-in: omit `review` and behavior is unchanged.

In [ ]:
from docuflow import commit_fill_async, preview_fill_async

review_pdf = OUTPUT_DIR / "blank_claim_review.pdf"
review_filled_pdf = OUTPUT_DIR / "filled_claim_review.pdf"
create_acroform_pdf(review_pdf)

# review=True prepares the fill but does NOT write the output PDF yet.
review_result = await fill_pdf_form_async(
    str(review_pdf),
    data=claim_data,
    output_path=str(review_filled_pdf),
    strategy="acroform",
    review=True,
)

print("committed:      ", review_result.committed)        # False - nothing written
print("review_status:  ", review_result.review_status)    # "pending"
print("needs_review:   ", review_result.needs_review)
print("review_reasons: ", review_result.review_reasons)
print("output written? ", review_filled_pdf.exists())     # False


### Preview the planned fill

`preview_fill_async` renders each affected page with the planned values overlaid (amber = edited or flagged, green = clean) and returns the saved image paths. No PDF is written — this is the backend a review UI would display.

In [ ]:
preview_dir = OUTPUT_DIR / "review_preview"
preview_images = await preview_fill_async(review_result, output_dir=str(preview_dir))
preview_images


### Correct, approve, and commit

`edit_field()` changes a planned value (or its placement) before anything is written, preserving the original value and recording a `FillCorrection`. `commit_fill_async()` requires an approved result (use `force=True` to bypass) and writes the output PDF.

In [ ]:
# Fix a planned value before committing.
review_result.edit_field(
    "policy_number",
    value="POL-999999",
    corrected_by="vannicola",
    reason="Corrected policy number from source",
)

review_result.approve(approved_by="vannicola")
await commit_fill_async(review_result)  # writes the PDF; returns the same result

print("committed:    ", review_result.committed)
print("review_status:", review_result.review_status)
print("output_path:  ", review_result.output_path)
print("output written?", review_filled_pdf.exists())
print("warnings:     ", review_result.warnings)
print("corrections:")
for corr in review_result.corrections:
    print(
        f"  {corr.field_name}: {corr.old_value!r} -> {corr.new_value!r} "
        f"by {corr.corrected_by!r} ({corr.reason!r})"
    )


In [ ]:
# Verify the approved values were written into the committed PDF.
reader = PdfReader(str(review_filled_pdf))
{name: field.get("/V") for name, field in reader.get_fields().items()}


## Result Object Summary

`FillingResult` is separate from `ExtractionResult` because filling writes trusted data into a document, while extraction reads data out of a document.

Useful fields:

- `result.success`
- `result.output_path`
- `result.strategy`
- `result.fields[field_name].value`
- `result.fields[field_name].formatted_value`
- `result.fields[field_name].method`
- `result.fields[field_name].bbox`
- `result.fields[field_name].placement.confidence`
- `result.unmapped_model_fields`
- `result.unmapped_pdf_fields`
- `result.warnings`
- `result.errors`


Review/approval fields (populated when `review=True`):

- `result.committed`
- `result.needs_review`
- `result.review_status`
- `result.review_reasons`
- `result.reviewed_by`, `result.reviewed_at`
- `result.rejection_reason`
- `result.corrections[i]` (`field_name`, `old_value`, `new_value`, `corrected_by`, `reason`)
